In [3]:
import numpy as np
import torch
from torch.distributions import Categorical
import torch.utils.data as data_utils
import matplotlib.pyplot as plt

import importlib

import AlphaBetaBot

importlib.reload(AlphaBetaBot)

# Import your classes
from AlphaBetaBot import AlphaBetaBot
from AlphaBetaBot import SolvedBot
from policy import Policy

import sys

from connect4 import Connect4Env, render_board
import gymnasium as gym

from utils import preprocess_board

In [4]:
def print_progress(iteration, total, prefix='', suffix='', decimals=1, length=40, fill='█'):
    """
    Call in a loop to create a terminal progress bar similar to TensorFlow/Keras.
    """
    percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = fill * filled_length + '-' * (length - filled_length)
    
    # \r returns the cursor to the start of the line
    sys.stdout.write(f'\r{prefix} |{bar}| {iteration}/{total} [{percent}%] {suffix}')
    sys.stdout.flush()
    
    # Print New Line on Complete
    if iteration == total:
        sys.stdout.write('\n')


def test_agent(env_class, policy:None|Policy|AlphaBetaBot|SolvedBot = None, policy_2: None|Policy|AlphaBetaBot|SolvedBot = None, games=100, visual=False) -> float: # (made with gemini)
    env = env_class()
    first_wins = 0
    
    print(f"Running {games} games against other Bot...")
    
    for game in range(games):
        obs, _ = env.reset()
        # AI plays as Player 1 (Blue)
        done = False
        
        while not done:
            if env.current_player == 1: # AI Turn
                if policy is not None and isinstance(policy, Policy):
                    ai_input = torch.from_numpy(preprocess_board(obs, 1)).float().unsqueeze(0).to("cuda")
                    with torch.no_grad():
                        logits = policy.forward(ai_input)[0]
                        mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)]).to("cuda")
                        action = torch.argmax(logits + mask).item()
                elif policy is not None and isinstance(policy, AlphaBetaBot):
                    action = policy.get_action(obs, 1)
                elif policy is not None and isinstance(policy, SolvedBot):
                    action = policy.get_action(board=obs, piece=1)
                    
                else:
                    # Random Logic
                    legal = [c for c in range(env.cols) if env.board[0][c] == 0]
                    action = np.random.choice(legal)
            else: # Random Turn
                if policy_2 is not None and isinstance(policy_2, Policy):
                    ai_input = torch.from_numpy(preprocess_board(obs, -1)).float().unsqueeze(0).to("cuda")
                    with torch.no_grad():
                        logits = policy_2.forward(ai_input)[0]
                        mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)]).to("cuda")
                        action = torch.argmax(logits + mask).item()
                elif policy_2 is not None and isinstance(policy_2, AlphaBetaBot):
                    action = policy_2.get_action(obs, -1)
                elif policy_2 is not None and isinstance(policy_2, SolvedBot):
                    action = policy_2.get_action(board=obs, piece=-1)
                    
                else:
                    # Random Logic
                    legal = [c for c in range(env.cols) if env.board[0][c] == 0]
                    action = np.random.choice(legal)
                
            obs, _, done, _, info = env.step(action)
            if visual:
                env.render()
            
        if info['winner'] == 1:
            if visual:
                print("Policy 1 Wins!")
            first_wins += 1
        elif info['winner'] == -1:
            if visual:
                print("Policy 2 Wins!")
            
    return first_wins/games

In [ ]:
# Define constants
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = Connect4Env()

ppo_policy = Policy(env.action_space.n, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, conv_layers_channels=[128, 64, 32], fc_layer_sizes=[512, 512, 256]).to(device)


# Training loop

In [ ]:
import connect4
import importlib
import time
importlib.reload(connect4)
from connect4 import Connect4Env


NUM_ENVS = 256
NUMBER_OF_GAMES_PER_UPDATE = 1024 # Collect at least this many steps before updating
TOTAL_UPDATES = 10000
MINIBATCH_SIZE = 512
EPOCHS = 4

envs = gym.vector.SyncVectorEnv([lambda: Connect4Env() for _ in range(NUM_ENVS)])

# Buffers to store incomplete episodes for each environment
# Structure: env_buffers[env_id] = list of (obs, action, logprob)
env_buffers = [[] for _ in range(NUM_ENVS)]

# observations structure:
# (batch_size, channels, height, width)

next_obs_np, _ = envs.reset()
next_obs = torch.tensor(preprocess_board(next_obs_np, env.current_player), dtype=torch.float32).to(device)

global_step = 0
global_games = 0
start_time = time.time()
current_start_time = time.time()

start_alpha_bot_level = 7

print(f"Starting training on {device} with {NUM_ENVS} parallel environments...")

for update in range(1, TOTAL_UPDATES + 1):
    
    # We collect until we have enough completed steps for a batch
    batch_data = {"states": [], "actions": [], "logprobs": [], "returns": []}

    games_played = 0
    
    while games_played < NUMBER_OF_GAMES_PER_UPDATE:
        global_step += NUM_ENVS
        
        with torch.no_grad():
            # Masking: Check top row (row 0) of the board (my pieces and enemy pieces)
            # If both are 0, the cell is empty (valid)
            # [all enviornments, the enemy and my channels, top row, all collumns]
            occupancy = next_obs[:, 0, 0, :] + next_obs[:, 1, 0, :]
            # returns an array that contains all the moves in each game and whether they are possible or not (if not -9_000_000_000, else 0)
            legal_mask = torch.where(occupancy == 0, 0.0, -1e9)
            
            logits = ppo_policy(next_obs)

            # nullify all illegal logits
            masked_logits = logits + legal_mask
            
            dist = Categorical(logits=masked_logits)
            action = dist.sample()
            logprob = dist.log_prob(action)
            # print(action.cpu().numpy())
            
        # Step the environments
        next_obs_np, _, terminated, _, infos = envs.step(action.cpu().numpy())
        
        
        # Store transitions in temporary buffers
        for i in range(NUM_ENVS):
            env_buffers[i].append({
                'state': next_obs[i].clone(), # Store the state BEFORE the step
                'action': action[i],
                'logprob': logprob[i]
            })
            
            # if i is done generate discounted rewards
            if terminated[i]:
                # Calculate Alternating Rewards
                episode_len = len(env_buffers[i])
                
                final_reward = 1.0
                if infos["winner"][i] == 0: # Draw
                    final_reward = 0.0
                
                episode_rewards = []
                gamma = 0.99
                curr = final_reward
                for j in range(episode_len):
                    episode_rewards.append(curr*(gamma**(episode_len-j-1)))
                    if curr != 0:
                        curr *= -1
                episode_rewards.reverse()
                
                # Add to main batch
                for t in range(episode_len):
                    batch_data['states'].append(env_buffers[i][t]['state'])
                    batch_data['actions'].append(env_buffers[i][t]['action'])
                    batch_data['logprobs'].append(env_buffers[i][t]['logprob'])
                    batch_data['returns'].append(episode_rewards[t])
                
                # Clear buffer for this env
                env_buffers[i] = []
                games_played+=1
                global_games+=1


        # Update next_obs for the next step
        next_obs = torch.tensor(preprocess_board(next_obs_np), dtype=torch.float32).to(device)

    # Optimization
    # Convert lists to tensors
    b_states = torch.stack(batch_data['states']).to(device)
    b_actions = torch.stack(batch_data['actions']).to(device)
    b_logprobs = torch.stack(batch_data['logprobs']).to(device)
    b_returns = torch.tensor(batch_data['returns'], dtype=torch.float32).to(device)

    b_advantages = ppo_policy.advantage(b_states, b_returns).detach()
    
    dataset = data_utils.TensorDataset(b_states, b_actions, b_logprobs, b_returns, b_advantages)
    loader = data_utils.DataLoader(dataset, batch_size=MINIBATCH_SIZE, shuffle=True)
    
    for epoch in range(EPOCHS):
        for batch in loader:
            # batch unpacks to: states, actions, old_probs, rewards, advantages
            ppo_policy.optimizer_step(*batch)


    if update % 100 == 0:
        win_rate_rand = test_agent(Connect4Env, ppo_policy,games=100)
        win_rate_alpha = test_agent(Connect4Env, ppo_policy, AlphaBetaBot(depth=9),games=1)
        if win_rate_alpha == 1:
            save_path = f"saves/bots/beat_alpha_beta_bot.pth"
            import os
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(ppo_policy.state_dict(), save_path)
            print(f"Saved model to {save_path} at delta-t: {time.time() - start_time}")
            test_agent(Connect4Env, ppo_policy, AlphaBetaBot(depth=9),games=1, visual=True)

        print(f"Update {update}/{TOTAL_UPDATES} | Global Step {global_step} | Rand Bot Win Rate: {win_rate_rand} | Alpha Beta Bot depth {start_alpha_bot_level} Win Rate: {win_rate_alpha} | Num Games: {global_games}")
        print("board moments before disaster:")
        print("Next move:", batch_data['actions'][-1])
        render_board(np.array(batch_data['states'][-1][0].cpu() - np.array(batch_data['states'][-1][1].cpu())))
        print(f"Time to train for 100 epochs: {time.time() - current_start_time}s")
        current_start_time = time.time()
    if update % 500 == 0:
        save_path = f"saves/bots/connect4_parallel_iter_{update}.pth"
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(ppo_policy.state_dict(), save_path)
        print(f"Saved model to {save_path} at delta-t: {time.time() - start_time}")
    # print("deleting")
    del b_states, b_actions, b_logprobs, b_returns, b_advantages, dataset, loader, batch_data # to prevent memory leaks
    torch.cuda.empty_cache()


Starting training on cuda with 256 parallel environments...
Running 100 games against other Bot...
Running 1 games against other Bot...
Update 100/10000 | Global Step 2739712 | Rand Bot Win Rate: 1.0 | Alpha Beta Bot depth 7 Win Rate: 0.0 | Num Games: 102924
board moments before disaster:
Next move: tensor(2, device='cuda:0')
              
              
              
              
              
              

Time to train for 100 epochs: 647.2947418689728s


C:\Users\yahli\AppData\Local\Temp\ipykernel_2876\4027844119.py:141: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  render_board(np.array(batch_data['states'][-1][0].cpu() - np.array(batch_data['states'][-1][1].cpu())))
C:\Users\yahli\AppData\Local\Temp\ipykernel_2876\4027844119.py:141: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  render_board(np.array(batch_data['states'][-1][0].cpu() - np.array(batch_data['states'][-1][1].cpu())))


In [9]:
test_agent(env_class=Connect4Env, policy=ppo_policy, policy_2=AlphaBetaBot(depth=9), games=1, visual=True)

Running 1 games against other Bot...
              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
        

0.0